## Exercise 1.4 Hotdog -- no hotdog
This is the poster hand-in project for the course. Please see the associated PDF for instructions.

In [ ]:
import os
import numpy as np
import glob
import PIL.Image as Image
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

We always check that we are running on a GPU

In [ ]:
if torch.cuda.is_available():
    print("The code will run on GPU.")
else:
    print("The code will run on CPU. Go to Edit->Notebook Settings and choose GPU as the hardware accelerator")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

We provide you with a class that can load the *hotdog/not hotdog* dataset you should use from /dtu/datasets1/02516/

In [ ]:
class Hotdog_NotHotdog(torch.utils.data.Dataset):
    def __init__(self, train, transform, data_path='hotdog_nothotdog'):
        'Initialization'
        self.transform = transform
        data_path = os.path.join(data_path, 'train' if train else 'test')
        image_classes = [os.path.split(d)[1] for d in glob.glob(data_path +'/*') if os.path.isdir(d)]
        image_classes.sort()
        self.name_to_label = {c: id for id, c in enumerate(image_classes)}
        self.image_paths = glob.glob(data_path + '/*/*.jpg')
        
    def __len__(self):
        'Returns the total number of samples'
        return len(self.image_paths)

    def __getitem__(self, idx):
        'Generates one sample of data'
        image_path = self.image_paths[idx]
        
        image = Image.open(image_path)
        c = os.path.split(os.path.split(image_path)[0])[1]
        y = self.name_to_label[c]
        X = self.transform(image)
        return X, y

Below is the simple way of converting the images to something that can be fed through a network.
Feel free to use something other than $128\times128$ images.

In [ ]:
# Load data
size = 128
train_transform = transforms.Compose([transforms.Resize((size, size)), 
                                    transforms.ToTensor()])
test_transform = transforms.Compose([transforms.Resize((size, size)), 
                                    transforms.ToTensor()])

batch_size = 64
trainset = Hotdog_NotHotdog(train=True, transform=train_transform)
train_loader = DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=0)
testset = Hotdog_NotHotdog(train=False, transform=test_transform)
test_loader = DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=0)

Let's look at some images from our data 

In [ ]:
images, labels = next(iter(train_loader))
plt.figure(figsize=(20,10))

for i in range(21):
    plt.subplot(5,7,i+1)
    plt.imshow(np.swapaxes(np.swapaxes(images[i].numpy(), 0, 2), 0, 1))
    plt.title(['hotdog', 'not hotdog'][labels[i].item()])
    plt.axis('off')


Now create a model and train it!


In [ ]:
# --- Lecture network baseline ---
class LectureNet(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.convolutional = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.fully_connected = nn.Sequential(
            nn.Linear(64*16*16, 128), nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.convolutional(x)
        x = x.view(x.size(0), -1)
        return self.fully_connected(x)


def train_model(model, train_loader, test_loader, optimizer, epochs=10):
    criterion = nn.CrossEntropyLoss()
    model.to(device)
    history = {'train_loss': [], 'test_acc': []}

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(X)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for X, y in test_loader:
                X, y = X.to(device), y.to(device)
                pred = model(X).argmax(1)
                correct += (pred == y).sum().item()
                total += y.size(0)

        history['train_loss'].append(total_loss / len(train_loader))
        history['test_acc'].append(correct / total)
        print(f"Epoch {epoch+1}: loss={history['train_loss'][-1]:.4f} acc={history['test_acc'][-1]:.4f}")

    return history

In [ ]:
lecture_model = LectureNet()
optimizer = torch.optim.Adam(lecture_model.parameters(), lr=1e-3)
lecture_history = train_model(lecture_model, train_loader, test_loader, optimizer, epochs=10)

In [ ]:
# --- ResNet-34 setup ---
import torchvision.models as models

def make_resnet(pretrained=True, use_bn=True, num_classes=2):
    weights = models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.resnet34(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, num_classes)

    if not use_bn:
        for name, module in model.named_modules():
            if isinstance(module, nn.BatchNorm2d):
                parent = model
                *path, last = name.split('.')
                for p in path:
                    parent = getattr(parent, p)
                setattr(parent, last, nn.Identity())

    return model

In [ ]:
resnet_train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])
resnet_train_transform_aug = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(20),
    transforms.ToTensor()
])
resnet_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

trainset_plain = Hotdog_NotHotdog(train=True, transform=resnet_train_transform)
trainset_aug = Hotdog_NotHotdog(train=True, transform=resnet_train_transform_aug)
testset_resnet = Hotdog_NotHotdog(train=False, transform=resnet_test_transform)

loader_plain = DataLoader(trainset_plain, batch_size=32, shuffle=True, num_workers=0)
loader_aug = DataLoader(trainset_aug, batch_size=32, shuffle=True, num_workers=0)
test_loader_resnet = DataLoader(testset_resnet, batch_size=32, shuffle=False, num_workers=0)

In [ ]:
# --- Full grid over ResNet-34 configs ---
results = {}

configs = []
for opt_name in ['adam', 'sgd']:
    for aug in [False, True]:
        for bn in [False, True]:
            for pretrained in [False, True]:
                if pretrained and not bn:
                    continue
                configs.append((opt_name, aug, bn, pretrained))

for opt_name, aug, bn, pretrained in configs:
    key = f"opt={opt_name}_aug={aug}_bn={bn}_pretrained={pretrained}"
    print(f"\n=== {key} ===")

    model = make_resnet(pretrained=pretrained, use_bn=bn)
    loader = loader_aug if aug else loader_plain

    if opt_name == 'adam':
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    else:
        optimizer = torch.optim.SGD(model.parameters(), lr=1e-3, momentum=0.9)

    history = train_model(model, loader, test_loader_resnet, optimizer, epochs=1)
    results[key] = history

In [ ]:
for key, h in results.items():
    print(f"{key}: final test acc = {h['test_acc'][-1]:.4f}")